## Iniciando o Spark e importando bibliotecas

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("AnaliseDadosBureau") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

bureau = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/bureau.csv")

bureau.createOrReplaceTempView("bureau")
bureau.show(5, truncate=False)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/13 13:26:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------+------------+-------------+---------------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------------+-------------------+--------------------+----------------------+---------------+------------------+-----------+
|SK_ID_CURR|SK_ID_BUREAU|CREDIT_ACTIVE|CREDIT_CURRENCY|DAYS_CREDIT|CREDIT_DAY_OVERDUE|DAYS_CREDIT_ENDDATE|DAYS_ENDDATE_FACT|AMT_CREDIT_MAX_OVERDUE|CNT_CREDIT_PROLONG|AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|AMT_CREDIT_SUM_LIMIT|AMT_CREDIT_SUM_OVERDUE|CREDIT_TYPE    |DAYS_CREDIT_UPDATE|AMT_ANNUITY|
+----------+------------+-------------+---------------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------------+-------------------+--------------------+----------------------+---------------+------------------+-----------+
|215354    |5714462     |Closed       |currency 1     |-497       |0                 |-153.0             |-153.0         

In [2]:
bureau.count()

1716428

In [3]:
bureau_balance = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/bureau_balance.csv")
bureau_balance.registerTempTable("bureau_balance")

bureau_balance.show(5, truncate=False)

/opt/spark/python/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


+------------+--------------+------+
|SK_ID_BUREAU|MONTHS_BALANCE|STATUS|
+------------+--------------+------+
|5715448     |0             |C     |
|5715448     |-1            |C     |
|5715448     |-2            |C     |
|5715448     |-3            |C     |
|5715448     |-4            |C     |
+------------+--------------+------+
only showing top 5 rows



In [4]:
bureau_balance.count()

27299925

#### Join entre as tabelas Bureau e Bureau Balance
- Renomeando algumas variáveis para facilitar o entendimento

In [5]:
bureau_bur_bal = spark.sql("""
                                Select
                                    a.SK_ID_CURR as PK_AGG_BUREAU,
                                    a.CREDIT_ACTIVE as FBC_CREDIT_ACTIVE_BUREAU,
                                    a.CREDIT_CURRENCY as FBC_CREDIT_CURRENCY_BUREAU,
                                    a.CREDIT_TYPE as FBC_CREDIT_TYPE_BUREAU,
                                    a.DAYS_CREDIT as FVL_DAYS_CREDIT_BUREAU,
                                    a.DAYS_CREDIT_ENDDATE as FVL_DAYS_CREDIT_ENDDATE_BUREAU,
                                    a.DAYS_CREDIT_UPDATE as FVL_DAYS_CREDIT_UPDATE_BUREAU,
                                    a.CREDIT_DAY_OVERDUE as FVL_CREDIT_DAY_OVERDUE_BUREAU,
                                    a.DAYS_ENDDATE_FACT as FVL_DAYS_ENDDATE_FACT_BUREAU,
                                    a.AMT_CREDIT_MAX_OVERDUE as FVL_AMT_CREDIT_MAX_OVERDUE_BUREAU,
                                    a.CNT_CREDIT_PROLONG as FVL_CNT_CREDIT_PROLONG_BUREAU,
                                    a.AMT_CREDIT_SUM_LIMIT as FVL_AMT_CREDIT_SUM_LIMIT_BUREAU,
                                    a.AMT_CREDIT_SUM_DEBT as FVL_AMT_CREDIT_SUM_DEBT_BUREAU,
                                    a.AMT_CREDIT_SUM as FVL_AMT_CREDIT_SUM_BUREAU,
                                    a.AMT_CREDIT_SUM_OVERDUE as FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU,
                                    a.AMT_ANNUITY as FVL_AMT_ANNUITY_BUREAU,
                                    cast(add_months('2023-12-01',b.MONTHS_BALANCE) as timestamp) as PK_DATREF,
                                    substr(translate(cast(add_months('2023-12-01',b.MONTHS_BALANCE) as string),'-',''),1,6) as PK_DATPART,
                                    b.STATUS as FBC_STATUS_BUREAU,
                                    b.MONTHS_BALANCE
                                from 
                                    bureau as a
                                left join 
                                    bureau_balance as b
                                  on 
                                    a.SK_ID_BUREAU = b.SK_ID_BUREAU
                            """)
# Retirando valores nulos
bureau_bur_bal = bureau_bur_bal.where(col("MONTHS_BALANCE").isNotNull())

# Filtrando somente histórico necessário (15 meses)
stage01 = bureau_bur_bal.where(col("MONTHS_BALANCE") >= -200)
stage01 = stage01.drop("MONTHS_BALANCE")

stage01.registerTempTable("stage01")
stage01.count()

24179741

In [6]:
spark.sql("""
            select
                PK_DATPART,
                PK_DATREF,
                count(*) as Volume
            from stage01
            group by 1,2
            order by  1 desc
""").show(200)

[Stage 22:=============================>                            (3 + 3) / 6]

+----------+-------------------+------+
|PK_DATPART|          PK_DATREF|Volume|
+----------+-------------------+------+
|    202312|2023-12-01 00:00:00|588522|
|    202311|2023-11-01 00:00:00|599342|
|    202310|2023-10-01 00:00:00|595803|
|    202309|2023-09-01 00:00:00|591398|
|    202308|2023-08-01 00:00:00|585261|
|    202307|2023-07-01 00:00:00|578586|
|    202306|2023-06-01 00:00:00|569988|
|    202305|2023-05-01 00:00:00|559284|
|    202304|2023-04-01 00:00:00|548832|
|    202303|2023-03-01 00:00:00|538856|
|    202302|2023-02-01 00:00:00|529230|
|    202301|2023-01-01 00:00:00|519793|
|    202212|2022-12-01 00:00:00|509680|
|    202211|2022-11-01 00:00:00|500070|
|    202210|2022-10-01 00:00:00|490812|
|    202209|2022-09-01 00:00:00|481518|
|    202208|2022-08-01 00:00:00|472392|
|    202207|2022-07-01 00:00:00|462390|
|    202206|2022-06-01 00:00:00|452337|
|    202205|2022-05-01 00:00:00|442211|
|    202204|2022-04-01 00:00:00|432536|
|    202203|2022-03-01 00:00:00|423022|


In [7]:
spark.sql("""
            select
                PK_DATPART,
                PK_DATREF,
                count(*) as Volume
            from stage01
            group by 1,2
            order by  1 desc
""").count()

97

In [8]:
stage01.show()

+-------------+------------------------+--------------------------+----------------------+----------------------+------------------------------+-----------------------------+-----------------------------+----------------------------+---------------------------------+-----------------------------+-------------------------------+------------------------------+-------------------------+---------------------------------+----------------------+-------------------+----------+-----------------+
|PK_AGG_BUREAU|FBC_CREDIT_ACTIVE_BUREAU|FBC_CREDIT_CURRENCY_BUREAU|FBC_CREDIT_TYPE_BUREAU|FVL_DAYS_CREDIT_BUREAU|FVL_DAYS_CREDIT_ENDDATE_BUREAU|FVL_DAYS_CREDIT_UPDATE_BUREAU|FVL_CREDIT_DAY_OVERDUE_BUREAU|FVL_DAYS_ENDDATE_FACT_BUREAU|FVL_AMT_CREDIT_MAX_OVERDUE_BUREAU|FVL_CNT_CREDIT_PROLONG_BUREAU|FVL_AMT_CREDIT_SUM_LIMIT_BUREAU|FVL_AMT_CREDIT_SUM_DEBT_BUREAU|FVL_AMT_CREDIT_SUM_BUREAU|FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU|FVL_AMT_ANNUITY_BUREAU|          PK_DATREF|PK_DATPART|FBC_STATUS_BUREAU|
+-------------

#### Salvando tabela particionada (Parquet)

In [9]:
nm_path = '/data/processed/bureau/'
stage01.write.partitionBy('PK_DATPART').parquet(nm_path, mode='overwrite')

In [10]:
test = spark.read.parquet("/data/processed/bureau")

test.createOrReplaceTempView("test")
test.show(5, truncate=False)

+-------------+------------------------+--------------------------+----------------------+----------------------+------------------------------+-----------------------------+-----------------------------+----------------------------+---------------------------------+-----------------------------+-------------------------------+------------------------------+-------------------------+---------------------------------+----------------------+-------------------+-----------------+----------+
|PK_AGG_BUREAU|FBC_CREDIT_ACTIVE_BUREAU|FBC_CREDIT_CURRENCY_BUREAU|FBC_CREDIT_TYPE_BUREAU|FVL_DAYS_CREDIT_BUREAU|FVL_DAYS_CREDIT_ENDDATE_BUREAU|FVL_DAYS_CREDIT_UPDATE_BUREAU|FVL_CREDIT_DAY_OVERDUE_BUREAU|FVL_DAYS_ENDDATE_FACT_BUREAU|FVL_AMT_CREDIT_MAX_OVERDUE_BUREAU|FVL_CNT_CREDIT_PROLONG_BUREAU|FVL_AMT_CREDIT_SUM_LIMIT_BUREAU|FVL_AMT_CREDIT_SUM_DEBT_BUREAU|FVL_AMT_CREDIT_SUM_BUREAU|FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU|FVL_AMT_ANNUITY_BUREAU|PK_DATREF          |FBC_STATUS_BUREAU|PK_DATPART|
+-------------

In [11]:
spark.stop()